# Analyze Example

### What this example demonstrates

This dataset intentionally includes a variety of common data quality issues so that the analysis plugins produce meaningful reports.

| Feature | Purpose |
|---------|---------|
| Missing values (`None`, `NaN`) | Demonstrates missing-value summaries and reports. |
| Placeholder values (`"N/A"`, `"None"`) | Demonstrates detection of potential missing-value placeholders. |
| Suspicious values (`"Unknown"`, `"Missing"`, `"?"`, `"-"`) | Demonstrates detection of suspicious missing-value indicators. |
| `"Yes"` / `"No"` column | Demonstrates potential boolean type detection. |
| Constant `Country` column | Demonstrates constant column detection. |
| Extreme values (`Age=150`, `Score=5`) | Demonstrates numeric outlier detection. |
| Rows with multiple missing values | Demonstrates sparse row reporting. |


### Notes

- `analyze()` is **non-destructive**. It does **not** modify the DataFrame, its values, column names, or data types.
- The method performs analysis only and stores the generated report in the DataFrame metadata.
- The analysis report can be accessed through:

```python
df.dg.report["analyze"]
```

In [3]:
import pandas as pd
import numpy as np
# notice the need to danda import to have accessor df
import danda # noqa: F401  # registers the pandas accessor
from danda.report_renderer import ReportRenderer

renderer = ReportRenderer()

# DataFrame
This dataset will trigger the following reports:

| Plugin                            | Trigger                                                               |
| --------------------------------- | --------------------------------------------------------------------- |
| **ColumnSummaryPlugin**           | Mixture of dtypes, missing values, varying cardinality                |
| **MissingValuesSummaryPlugin**    | `Name`, `Age`, `Country`, `Score`, `Comments` contain missing values  |
| **MissingValuesReportPlugin**     | Several columns have missing values                                   |
| **PotentialMissingValuesPlugin**  | `"N/A"`, `"None"`, empty string (`""`) *(depending on configuration)* |
| **SuspiciousMissingValuesPlugin** | `"Unknown"`, `"Missing"`, `"?"`, `"-"`                                |
| **SparseRowsReportPlugin**        | Row with multiple missing values (especially row 4)                   |
| **PotentialBooleanTypePlugin**    | `"Subscribed"` contains only `"Yes"`/`"No"`                           |
| **ConstantColumnsPlugin**         | `"Country"` has only one non-null value (`"UK"`)                      |
| **OutlierReportPlugin**           | `Age=150` and `Score=5` are clear outliers                            |


In [5]:
df = pd.DataFrame({
    "CustomerID": [101, 102, 103, 104, 105, 106, 107, 108],
    "Name": [
        "Alice", "Bob", "Unknown", "David",
        None, "Frank", "Grace", "N/A"
    ],
    "Age": [25, 32, 29, 31, np.nan, 28, 150, 30],
    "City": [
        "London", "Paris", "None", "Berlin",
        "Paris", "?", "-", "London"
    ],
    "Subscribed": [
        "Yes", "No", "Yes", "No",
        "Yes", "No", "Yes", "No"
    ],
    "Country": [
        "UK", "UK", "UK", "UK",
        "UK", "UK", "UK", None
    ],
    "Score": [82, 79, 81, 84, 80, np.nan, 5, 83],
    "Comments": [
        "Good", "", "Missing", None,
        "Excellent", "Good", "Unknown", "Good"
    ]
})

df

,CustomerID,Name,Age,City,Subscribed,Country,Score,Comments
0,101,Alice,25.0,London,Yes,UK,82.0,Good
1,102,Bob,32.0,Paris,No,UK,79.0,
2,103,Unknown,29.0,None,Yes,UK,81.0,Missing
3,104,David,31.0,Berlin,No,UK,84.0,NaN
4,105,NaN,NaN,Paris,Yes,UK,80.0,Excellent
5,106,Frank,28.0,?,No,UK,NaN,Good
6,107,Grace,150.0,-,Yes,UK,5.0,Unknown
7,108,N/A,30.0,London,No,NaN,83.0,Good


### Expected report contents


The generated report will include sections similar to:

- **Column summary**
  - Data type
  - Missing value count and percentage
  - Number of unique values

- **Missing value summary**
  - Total missing cells
  - Rows containing missing values
  - Complete rows

- **Missing value report**
  - Per-column missing counts

- **Potential missing values**
  - Placeholder values such as `"N/A"` and `"None"`

- **Suspicious missing values**
  - Indicators such as `"Unknown"`, `"Missing"`, `"?"`, and `"-"`

- **Sparse rows**
  - Rows with a high proportion of missing values

- **Potential boolean columns**
  - Columns containing two non-standard boolean values (for example `"Yes"` and `"No"`)

- **Constant columns**
  - Columns containing a single unique non-missing value

- **Outlier report**
  - Numeric columns containing statistically significant outliers

In [7]:
report = df.dg.analyze().dg.report
print(renderer.render(report))

Danda Report

Analyze
-------

ColumnSummaryPlugin
    Column Summary:
    
    CustomerID
    Type: int64
    Missing: 0 (0%)
    Unique: 8
    
    Name
    Type: str
    Missing: 1 (12%)
    Unique: 7
    
    Age
    Type: float64
    Missing: 1 (12%)
    Unique: 7
    
    City
    Type: str
    Missing: 0 (0%)
    Unique: 6
    
    Subscribed
    Type: str
    Missing: 0 (0%)
    Unique: 2
    
    Country
    Type: str
    Missing: 1 (12%)
    Unique: 1
    
    Score
    Type: float64
    Missing: 1 (12%)
    Unique: 7
    
    Comments
    Type: str
    Missing: 1 (12%)
    Unique: 5

MissingValuesSummaryPlugin
    Missing Value Summary
    Rows: 8
    Columns: 8
    
    Missing cells: 5 (7.8%)
    Columns with missing: 5
    Rows with missing: 4
    Complete rows: 4 (50.0%)

MissingValuesReportPlugin
    Missing values detected:
    - Age: 1 (12.5%)
    - Comments: 1 (12.5%)
    - Country: 1 (12.5%)
    - Name: 1 (12.5%)
    - Score: 1 (12.5%)

PotentialMissingValuesPlugin
